# Step 4: RQ2 Top-4 Overlap Summary

This notebook summarizes the RQ2 Top-4 condition without modifying any existing RQ2 notebooks or result files. It reads the existing `Top_4` batch outputs, compares each LLM-returned Top-4 feature set with the model-grounded reference Top-4 feature set, and reports `Overlap@4`.

Outputs are written only to a new folder: `Result/LLM/RQ2_Top4_Overlap_Summary/`.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd()
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending")


def find_local_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Data" / "ModifiedData").exists() and (candidate / "Result").exists():
            return candidate
    return start.parent if start.name == "Code" else start


BASE_DIR = COLAB_PROJECT_ROOT if COLAB_PROJECT_ROOT.exists() else find_local_project_root(PROJECT_ROOT)
RESULT_ROOT = BASE_DIR / "Result"
RESULT_PROJECT_DIR = RESULT_ROOT / "LendingClub" if (RESULT_ROOT / "LendingClub").exists() else RESULT_ROOT
SHAP_DIR = RESULT_PROJECT_DIR / "SHAP"
SHAP_RETRAINED_DIR = RESULT_PROJECT_DIR / "SHAP_retrained"
LLM_DIR = RESULT_PROJECT_DIR / "LLM"
SUMMARY_DIR = LLM_DIR / "RQ2_Top4_Overlap_Summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 4

print(f"Base directory: {BASE_DIR}")
print(f"LLM directory: {LLM_DIR}")
print(f"Summary output directory: {SUMMARY_DIR}")


## 2. Feature Name Mapping

The batch files usually already contain canonical raw feature names in `Feature`, but this mapping keeps the summary robust to readable feature names.

In [ ]:
FEATURE_TO_NAME_MAP = {
    "home_ownership": "Home Ownership Status",
    "verification_status": "Income Verification Status",
    "purpose": "Loan Purpose",
    "area": "Borrower Area",
    "addr_state": "Borrower State",
    "term": "Number of Payments",
    "grade": "Loan Grade",
    "sub_grade": "Loan Sub Grade",
    "emp_length": "Employment Length",
    "pub_rec_bankruptcies": "Number of Public Bankruptcies",
    "funded_amnt": "Loan Amount",
    "int_rate": "Interest Rate",
    "installment": "Monthly Payment",
    "annual_inc": "Annual Income",
    "dti": "Debt to Income Ratio",
    "delinq_2yrs": "Number of Delinquencies in Past 2 Years",
    "fico_range_low": "Lowest FICO Score",
    "fico_range_high": "Highest FICO Score",
    "inq_last_6mths": "Number of Inquiries in Last 6 Months",
    "open_acc": "Number of Open Accounts",
    "pub_rec": "Number of Derogatory Public Records",
    "revol_bal": "Revolving Balance",
    "revol_util": "Revolving Utilization Rate",
    "total_acc": "Total Number of Accounts",
}
NAME_TO_FEATURE_MAP = {v: k for k, v in FEATURE_TO_NAME_MAP.items()}


## 3. RQ2 Top-4 Result Folders

The main RQ2 redesigned folders are included by default. Few-shot RQ2 folders are also included if they exist and are labeled separately as `FewShot`. Remove those entries from `RUN_SPECS` if you only want the zero-shot RQ2 table.

In [ ]:
RUN_SPECS = [
    {
        "Experiment": "ZeroShot",
        "BaseModel": "Logistic Regression",
        "Top4Dir": LLM_DIR / "RQ2_Logistic_Redesigned" / "Top_4",
        "SamplePath": LLM_DIR / "500_samples_logistic.xlsx",
        "ReferencePath": SHAP_DIR / "Logistic_Contrib.xlsx",
        "ReferenceType": "wide_abs",
    },
    {
        "Experiment": "ZeroShot",
        "BaseModel": "XGBoost",
        "Top4Dir": LLM_DIR / "RQ2_XGBoost_Redesigned" / "Top_4",
        "SamplePath": LLM_DIR / "500_samples_xgboost_matched.xlsx",
        "ReferencePath": SHAP_RETRAINED_DIR / "XGBoost_SHAP_Aggregated.xlsx",
        "ReferenceType": "long_abs",
    },
    {
        "Experiment": "FewShot",
        "BaseModel": "Logistic Regression",
        "Top4Dir": LLM_DIR / "RQ2_Logistic_FewShot_Redesigned" / "Top_4",
        "SamplePath": LLM_DIR / "500_samples_logistic.xlsx",
        "ReferencePath": SHAP_DIR / "Logistic_Contrib.xlsx",
        "ReferenceType": "wide_abs",
    },
    {
        "Experiment": "FewShot",
        "BaseModel": "XGBoost",
        "Top4Dir": LLM_DIR / "RQ2_XGBoost_FewShot_Redesigned" / "Top_4",
        "SamplePath": LLM_DIR / "500_samples_xgboost_matched.xlsx",
        "ReferencePath": SHAP_RETRAINED_DIR / "XGBoost_SHAP_Aggregated.xlsx",
        "ReferenceType": "long_abs",
    },
]

for spec in RUN_SPECS:
    status = "FOUND" if spec["Top4Dir"].exists() else "MISSING"
    print(f"{status}: {spec['Experiment']} | {spec['BaseModel']} | {spec['Top4Dir']}")


## 4. Helper Functions

In [ ]:
def normalize_feature_text(text):
    text = str(text).strip()
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()


def feature_aliases(feature):
    aliases = {str(feature)}
    if feature in FEATURE_TO_NAME_MAP:
        aliases.add(FEATURE_TO_NAME_MAP[feature])
    if feature in NAME_TO_FEATURE_MAP:
        aliases.add(NAME_TO_FEATURE_MAP[feature])
    return {alias for alias in aliases if alias is not None and str(alias).strip()}


def build_feature_lookup(allowed_features):
    lookup = {}
    for feature in allowed_features:
        for alias in feature_aliases(feature):
            lookup[normalize_feature_text(alias)] = feature
    return lookup


def canonicalize_feature(feature, allowed_features):
    lookup = build_feature_lookup(allowed_features)
    normalized = normalize_feature_text(feature)
    if normalized in lookup:
        return lookup[normalized]
    for alias_key, raw_feature in lookup.items():
        if alias_key and alias_key in normalized:
            return raw_feature
    return None


def load_sample_df(sample_path):
    sample_df = pd.read_excel(sample_path)
    sample_df["Rowindex"] = sample_df["Rowindex"].astype(int)
    return sample_df


def load_reference(spec):
    if spec["ReferenceType"] == "wide_abs":
        ref = pd.read_excel(spec["ReferencePath"], index_col=0)
        ref.index = ref.index.astype(int)
        return ref

    ref = pd.read_excel(spec["ReferencePath"])
    ref["Rowindex"] = ref["Rowindex"].astype(int)
    if "AbsSHAPValue" not in ref.columns:
        if "AbsContribution" in ref.columns:
            ref["AbsSHAPValue"] = ref["AbsContribution"].abs()
        elif "SignedContribution" in ref.columns:
            ref["AbsSHAPValue"] = ref["SignedContribution"].abs()
        else:
            raise KeyError("Long reference table needs AbsSHAPValue, AbsContribution, or SignedContribution.")
    return ref


def reference_topk(rowindex, reference_df, reference_type, k=TOP_K):
    rowindex = int(rowindex)
    if reference_type == "wide_abs":
        row = reference_df.loc[rowindex]
        return row.abs().sort_values(ascending=False).head(k).index.astype(str).tolist()

    row = reference_df[reference_df["Rowindex"] == rowindex].copy()
    if row.empty:
        raise KeyError(f"Rowindex {rowindex} is missing from reference table.")
    return row.sort_values("AbsSHAPValue", ascending=False).head(k)["Feature"].astype(str).tolist()


def sorted_batch_files(model_dir):
    def batch_num(path):
        match = re.search(r"batch_(\d+)", path.stem)
        return int(match.group(1)) if match else 10**9
    return sorted(model_dir.glob("batch_*.xlsx"), key=batch_num)


def load_llm_batches(model_dir):
    frames = []
    for path in sorted_batch_files(model_dir):
        frame = pd.read_excel(path)
        frame["SourceFile"] = path.name
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["Rowindex", "FeatureRank", "Feature", "ReadableFeature", "SourceFile"])
    df = pd.concat(frames, ignore_index=True)
    df["Rowindex"] = df["Rowindex"].astype(int)
    df["FeatureRank"] = pd.to_numeric(df["FeatureRank"], errors="coerce")
    return df.sort_values(["Rowindex", "FeatureRank", "SourceFile"]).reset_index(drop=True)


def load_error_count(model_dir):
    error_path = model_dir / "errors.jsonl"
    if not error_path.exists():
        return 0
    count = 0
    with open(error_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
    return count


def overlap_at_k(llm_features, ref_features, k=TOP_K):
    return len(set(llm_features[:k]) & set(ref_features[:k])) / k


## 5. Compute Overlap@4 by Sample

This recomputes `Overlap@4` from the saved Top-4 batch files. Existing `metric.xlsx` files are not read or overwritten.

In [ ]:
def evaluate_top4_overlap_for_spec(spec):
    if not spec["Top4Dir"].exists():
        print(f"Skipping missing Top_4 directory: {spec['Top4Dir']}")
        return pd.DataFrame()

    sample_df = load_sample_df(spec["SamplePath"])
    reference_df = load_reference(spec)
    rows = []

    for model_dir in sorted([p for p in spec["Top4Dir"].iterdir() if p.is_dir()]):
        llm_df = load_llm_batches(model_dir)
        error_count = load_error_count(model_dir)

        for rowindex in sample_df["Rowindex"].astype(int):
            ref_top4 = reference_topk(rowindex, reference_df, spec["ReferenceType"], TOP_K)
            allowed_features = reference_topk(rowindex, reference_df, spec["ReferenceType"], k=24)
            model_rows = llm_df[llm_df["Rowindex"] == rowindex].sort_values("FeatureRank")

            raw_features = model_rows["Feature"].astype(str).tolist()
            canonical_features = []
            unknown_features = []
            for feature in raw_features:
                mapped = canonicalize_feature(feature, allowed_features)
                if mapped is None:
                    unknown_features.append(feature)
                else:
                    canonical_features.append(mapped)

            returned_top4 = canonical_features[:TOP_K]
            duplicate_count = len(returned_top4) - len(set(returned_top4))
            parsed = len(returned_top4) > 0
            complete_top4 = len(returned_top4) == TOP_K
            overlap = overlap_at_k(returned_top4, ref_top4, TOP_K) if parsed else np.nan
            hit_count = int(round(overlap * TOP_K)) if parsed else np.nan

            rows.append({
                "Experiment": spec["Experiment"],
                "BaseModel": spec["BaseModel"],
                "LLMResultDir": model_dir.name,
                "Rowindex": rowindex,
                "Parsed": parsed,
                "ReturnedCount": len(returned_top4),
                "CompleteTop4": complete_top4,
                "UnknownFeatureCount": len(unknown_features),
                "DuplicateTop4Count": duplicate_count,
                "OverlapAt4": overlap,
                "HitCountAt4": hit_count,
                "ReferenceTop4": "; ".join(ref_top4),
                "LLMTop4": "; ".join(returned_top4),
                "UnknownFeatures": "; ".join(unknown_features),
                "ModelErrorLogCount": error_count,
            })

    return pd.DataFrame(rows)


all_detail_frames = []
for spec in RUN_SPECS:
    detail = evaluate_top4_overlap_for_spec(spec)
    if not detail.empty:
        all_detail_frames.append(detail)

if all_detail_frames:
    top4_detail_df = pd.concat(all_detail_frames, ignore_index=True)
else:
    top4_detail_df = pd.DataFrame()

print(f"Sample-level rows: {len(top4_detail_df)}")
display(top4_detail_df.head())


## 6. Summary Table

In [ ]:
def summarize_top4_overlap(detail_df):
    if detail_df.empty:
        return pd.DataFrame()

    rows = []
    for keys, group in detail_df.groupby(["Experiment", "BaseModel", "LLMResultDir"], dropna=False):
        experiment, base_model, llm_dir = keys
        values = pd.to_numeric(group["OverlapAt4"], errors="coerce").dropna()
        rows.append({
            "Experiment": experiment,
            "BaseModel": base_model,
            "LLMResultDir": llm_dir,
            "N_Total": len(group),
            "N_Parsed": int(group["Parsed"].sum()),
            "ParsedRate": group["Parsed"].mean(),
            "CompleteTop4Rate": group["CompleteTop4"].mean(),
            "MeanOverlapAt4": values.mean() if len(values) else np.nan,
            "MedianOverlapAt4": values.median() if len(values) else np.nan,
            "StdOverlapAt4": values.std(ddof=1) if len(values) > 1 else np.nan,
            "MinOverlapAt4": values.min() if len(values) else np.nan,
            "MaxOverlapAt4": values.max() if len(values) else np.nan,
            "PerfectOverlapRate": (values == 1).mean() if len(values) else np.nan,
            "ZeroOverlapRate": (values == 0).mean() if len(values) else np.nan,
            "MeanHitCountAt4": pd.to_numeric(group["HitCountAt4"], errors="coerce").mean(),
            "TotalUnknownFeatureCount": int(group["UnknownFeatureCount"].sum()),
            "TotalDuplicateTop4Count": int(group["DuplicateTop4Count"].sum()),
            "TotalModelErrorLogCount": int(group["ModelErrorLogCount"].max()) if len(group) else 0,
        })

    return pd.DataFrame(rows).sort_values(["Experiment", "BaseModel", "LLMResultDir"]).reset_index(drop=True)


top4_summary_df = summarize_top4_overlap(top4_detail_df)
display(top4_summary_df)


## 7. Distribution Table

This table shows how many samples had 0, 1, 2, 3, or 4 overlapping features in the Top-4 set.

In [ ]:
if top4_detail_df.empty:
    top4_distribution_df = pd.DataFrame()
else:
    top4_distribution_df = (
        top4_detail_df
        .dropna(subset=["HitCountAt4"])
        .assign(HitCountAt4=lambda df: df["HitCountAt4"].astype(int))
        .groupby(["Experiment", "BaseModel", "LLMResultDir", "HitCountAt4"])
        .size()
        .rename("Count")
        .reset_index()
    )
    top4_distribution_df["Rate"] = top4_distribution_df.groupby(["Experiment", "BaseModel", "LLMResultDir"])["Count"].transform(lambda s: s / s.sum())

display(top4_distribution_df)


In [ ]:
def make_appendix_distribution_count_table(top4_distribution_df, experiment):
    df = top4_distribution_df[
        top4_distribution_df["Experiment"] == experiment
    ].copy()

    table = df.pivot_table(
        index=["BaseModel", "LLMResultDir"],
        columns="HitCountAt4",
        values="Count",
        aggfunc="sum",
        fill_value=0,
    )

    for k in range(5):
        if k not in table.columns:
            table[k] = 0

    table = table[[0, 1, 2, 3, 4]]
    table.columns = [f"{k} hits" for k in table.columns]
    table = table.reset_index()

    hit_cols = [f"{k} hits" for k in range(5)]
    table["N"] = table[hit_cols].sum(axis=1)

    table = table[["BaseModel", "LLMResultDir", "N"] + hit_cols]

    return table


zeroshot_appendix_count = make_appendix_distribution_count_table(
    top4_distribution_df,
    "ZeroShot",
)

fewshot_appendix_count = make_appendix_distribution_count_table(
    top4_distribution_df,
    "FewShot",
)

display(zeroshot_appendix_count)


In [ ]:
display(fewshot_appendix_count)

## 8. Save New Summary Outputs

These files are new summary artifacts only. They do not overwrite any existing RQ2 results.

In [ ]:
detail_path = SUMMARY_DIR / "rq2_top4_overlap_by_sample.xlsx"
summary_path = SUMMARY_DIR / "rq2_top4_overlap_summary.xlsx"
distribution_path = SUMMARY_DIR / "rq2_top4_overlap_distribution.xlsx"

if not top4_detail_df.empty:
    top4_detail_df.to_excel(detail_path, index=False)
    top4_summary_df.to_excel(summary_path, index=False)
    top4_distribution_df.to_excel(distribution_path, index=False)

    print(f"Saved sample-level file: {detail_path}")
    print(f"Saved summary file: {summary_path}")
    print(f"Saved distribution file: {distribution_path}")
else:
    print("No Top-4 detail rows to save.")


## 9. Manuscript-Ready Table

A compact version for the paper. Rename `LLMResultDir` labels as needed for manuscript formatting.

In [ ]:
if top4_summary_df.empty:
    report_table = pd.DataFrame()
else:
    report_cols = [
        "Experiment", "BaseModel", "LLMResultDir", "N_Total", "N_Parsed", "ParsedRate",
        "MeanOverlapAt4", "MedianOverlapAt4", "PerfectOverlapRate", "ZeroOverlapRate",
        "MeanHitCountAt4", "CompleteTop4Rate",
    ]
    report_table = top4_summary_df[report_cols].copy()
    for col in ["ParsedRate", "MeanOverlapAt4", "MedianOverlapAt4", "PerfectOverlapRate", "ZeroOverlapRate", "MeanHitCountAt4", "CompleteTop4Rate"]:
        report_table[col] = report_table[col].round(4)
    display(report_table)

    report_path = SUMMARY_DIR / "rq2_top4_overlap_report_table.xlsx"
    report_table.to_excel(report_path, index=False)
    print(f"Saved manuscript-ready table: {report_path}")


In [ ]:
report_table[(report_table.BaseModel == "XGBoost")&(report_table.Experiment == "ZeroShot")]

## 10. Bootstrap 95% CI for Mean Overlap@4

This section estimates instance-level bootstrap confidence intervals for the mean of `Overlap@4`. Within each experiment, base model, and LLM result directory, it resamples evaluated instances with replacement and recomputes the mean overlap. The confidence interval is the percentile interval from the bootstrap distribution.

In [ ]:
def bootstrap_mean_ci(values, n_bootstrap=5000, ci=0.95, random_state=42):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    n = len(values)
    if n == 0:
        return {
            "MeanOverlapAt4": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "N_Parsed": 0,
            "BootstrapSamples": n_bootstrap,
            "CI_Level": ci,
        }

    rng = np.random.default_rng(random_state)
    alpha = 1 - ci
    boot_idx = rng.integers(0, n, size=(n_bootstrap, n))
    boot_means = values[boot_idx].mean(axis=1)
    lower, upper = np.quantile(boot_means, [alpha / 2, 1 - alpha / 2])

    return {
        "MeanOverlapAt4": float(values.mean()),
        "CI_Lower": float(lower),
        "CI_Upper": float(upper),
        "N_Parsed": n,
        "BootstrapSamples": n_bootstrap,
        "CI_Level": ci,
    }


def build_overlap4_bootstrap_ci(detail_df, n_bootstrap=5000, ci=0.95, random_state=42):
    if detail_df.empty:
        return pd.DataFrame()

    rows = []
    group_cols = ["Experiment", "BaseModel", "LLMResultDir"]
    for group_id, group in detail_df.groupby(group_cols, dropna=False):
        experiment, base_model, llm_dir = group_id
        ci_result = bootstrap_mean_ci(
            group["OverlapAt4"],
            n_bootstrap=n_bootstrap,
            ci=ci,
            random_state=random_state,
        )
        rows.append({
            "Experiment": experiment,
            "BaseModel": base_model,
            "LLMResultDir": llm_dir,
            **ci_result,
        })

    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


bootstrap_ci_df = build_overlap4_bootstrap_ci(
    top4_detail_df,
    n_bootstrap=5000,
    ci=0.95,
    random_state=42,
)

display(bootstrap_ci_df)

bootstrap_ci_path = SUMMARY_DIR / "rq2_top4_overlap_mean_bootstrap_ci.xlsx"
bootstrap_ci_df.to_excel(bootstrap_ci_path, index=False)
print(f"Saved bootstrap CI file: {bootstrap_ci_path}")


In [ ]:
bootstrap_ci_df[(bootstrap_ci_df.BaseModel == "XGBoost") & (bootstrap_ci_df.Experiment == "ZeroShot")]